# Effective dimension of single-vector datasets

Participation ratio (PR) of every dataset in `NPY_DATASETS`. Each is a directory holding `vectors.npy` (`n x d`, one point per row), `ids.npy` and `sq_norms.npy`. `ids.npy` is not needed. `sq_norms.npy` is checked against the first rows of `vectors.npy` and a warning is printed if they don't match.

For centred points $x_i$ with covariance $C$, $\mathrm{PR} = (\operatorname{tr} C)^2 / \operatorname{tr}(C^2) = (\sum_k \lambda_k)^2 / \sum_k \lambda_k^2$. It ranges from 1 (one direction) to $d$ (isotropic). The computation lives in `effdim.py`: an exact one-pass accumulator with O(d²) memory (same value as `participation_ratio`), reading memory-mapped vectors in blocks, so it scales to billion-point sets. With `SAMPLE = m` it also estimates PR on `SAMPLE_SEEDS` random subsets of m points (mean ± sd); `EXACT = False` runs the sampling estimate alone.

For the OBLIQ multi-vector datasets (mean pooled), see `effective_dim_obliq.ipynb`. Use the **aws** kernel.

In [ ]:
from pathlib import Path

import pandas as pd

from effdim import estimate_pr, open_npy_dataset, row_blocks

NPY_DATASETS = []    # directories with vectors.npy (+ ids.npy, sq_norms.npy), e.g. ["/scratch/<netid>/deep1b"]
CHUNK = 1 << 16      # rows read per block
EXACT = True         # full one-pass PR over every point
SAMPLE = None        # e.g. 1_000_000: also estimate PR from random subsets of this many points
SAMPLE_SEEDS = 5
SEED = 0

## Compute

In [ ]:
rows = []
for folder in map(Path, NPY_DATASETS):
    V = open_npy_dataset(folder)
    pr = estimate_pr(len(V), lambda items: row_blocks(V, items, CHUNK), folder.name,
                     EXACT, SAMPLE, SAMPLE_SEEDS, SEED)
    rows.append({"dataset": folder.name, "dtype": str(V.dtype), "dim": V.shape[1], "points": len(V), **pr})

effdim = pd.DataFrame(rows)
for c in ("PR", "sample PR mean"):
    if c in effdim:
        effdim[c + " / d"] = effdim[c] / effdim["dim"]

## Results

`/ d` columns normalise by the embedding dimension, so datasets of different dimension can be compared. Sample columns appear only when `SAMPLE` is set and smaller than the number of points.

In [ ]:
effdim.style.format({c: "{:.1f}" for c in effdim.columns if "PR" in c and "/ d" not in c}
                    | {c: "{:.3f}" for c in effdim.columns if "/ d" in c}
                    | {"points": "{:,}", "sample size": "{:,.0f}"}, na_rep="–").hide(axis="index")

In [ ]:
# effdim.to_csv("../results/effective_dimension.csv", index=False)